# SpectraGuard external evaluation

This notebook evaluates existing checkpoints without retraining. The two tasks remain independent:

- `face_dataset_external_test`: an exploratory external test of the FaceForensics++ RGB checkpoint. The supplied face CSV is verified as AI-generated versus authentic imagery, not assumed to be a traditional face-swap benchmark.
- `ai_generated`: cross-dataset evaluation of CIFAKE RGB, FFT, and RGB+FFT Fusion checkpoints on the Kaggle AI-vs-human dataset.

All inputs, label mappings, model loading, metrics, predictions, reports, and plots use the repository implementation in `backend.training.external_evaluation`.

In [9]:
from pathlib import Path
import json
import platform
import sys

PROJECT_ROOT = Path(r"E:\Projects\Deepfake detector")
if not (PROJECT_ROOT / "backend" / "training").is_dir():
    raise FileNotFoundError(f"Invalid SpectraGuard project root: {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_ROOT = PROJECT_ROOT / "external_datasets"
FACE_ROOT = DATASET_ROOT / "face_deepfake"
AI_ROOT = DATASET_ROOT / "ai_generated"
OUTPUT_ROOT = PROJECT_ROOT / "evaluation_results"
for directory in (FACE_ROOT, AI_ROOT, OUTPUT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 8
SAMPLE_PER_CLASS = 100
print("Project:", PROJECT_ROOT)
print("Python:", platform.python_version())

Project: E:\Projects\Deepfake detector
Python: 3.11.9


In [10]:
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from backend.app.frequency import preprocess_fft
from backend.app.preprocessing import preprocess_rgb
from backend.training.external_evaluation import (
    CLASS_NAMES, ModelSpec, balanced_sample, build_ai_manifest, build_face_manifest,
    calculate_model_metrics, classification_report_dict, download_kaggle_dataset,
    export_results, inspect_dataset, load_evaluation_model, plot_curves, predict_model,
    safe_extract_zip, bootstrap_intervals,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

PyTorch: 2.5.1+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM GB: 6.0


## Dataset download and safe extraction

Run this cell only when the Kaggle archives are not already cached. It never prints credentials and uses the dataset download API, not competition download.

In [11]:
DATASET_IDS = {
    "face": "chuneeb/deepfake-detection-dataset-2026",
    "ai": "alessandrasala79/ai-vs-human-generated-dataset",
}

def ensure_dataset(dataset_id, root):
    archive = download_kaggle_dataset(dataset_id, root)
    extracted = root / f"{archive.stem}_extracted"
    if not any(extracted.rglob("*.csv")) and not any(extracted.rglob("*.jpg")):
        safe_extract_zip(archive, extracted)
    return archive, extracted

# Uncomment to download when Kaggle authentication is configured:
# FACE_ARCHIVE, FACE_EXTRACTED = ensure_dataset(DATASET_IDS["face"], FACE_ROOT)
# AI_ARCHIVE, AI_EXTRACTED = ensure_dataset(DATASET_IDS["ai"], AI_ROOT)
print("Cached roots:", FACE_ROOT, AI_ROOT)

Cached roots: E:\Projects\Deepfake detector\external_datasets\face_deepfake E:\Projects\Deepfake detector\external_datasets\ai_generated


## Dataset inspection and manifests

The face dataset is qualified from its metadata before use. The AI-vs-human dataset uses its observed `train.csv` contract (`file_name`, `label`) and refuses unknown schemas.

In [12]:
FACE_EXTRACTED = next((p for p in FACE_ROOT.glob("*_extracted") if p.is_dir()), FACE_ROOT)
AI_EXTRACTED = next((p for p in AI_ROOT.glob("*_extracted") if p.is_dir()), AI_ROOT)
FACE_INSPECTION = inspect_dataset(FACE_EXTRACTED)
AI_INSPECTION = inspect_dataset(AI_EXTRACTED)
print("Face files/images/CSVs:", len(FACE_INSPECTION["files"]), len(FACE_INSPECTION["image_files"]), len(FACE_INSPECTION["csv_files"]))
print("AI files/images/CSVs:", len(AI_INSPECTION["files"]), len(AI_INSPECTION["image_files"]), len(AI_INSPECTION["csv_files"]))
for info in FACE_INSPECTION["csv_info"] + AI_INSPECTION["csv_info"]:
    print(info["path"].relative_to(PROJECT_ROOT), info["columns"])

Face files/images/CSVs: 1 0 1
AI files/images/CSVs: 85492 85490 2
external_datasets\face_deepfake\deepfake-detection-dataset-2026_extracted\FINAL_DATASET.csv ['image_id', 'image_url', 'label', 'label_numeric', 'category', 'gender', 'age_group', 'source', 'fake_method', 'image_quality', 'resolution', 'confidence_score', 'detection_difficulty', 'dataset_split', 'date_collected', 'version', 'year']
external_datasets\ai_generated\ai-vs-human-generated-dataset_extracted\test.csv ['id']
external_datasets\ai_generated\ai-vs-human-generated-dataset_extracted\train.csv ['Unnamed: 0', 'file_name', 'label']


In [13]:
FACE_CSV = FACE_EXTRACTED / "FINAL_DATASET.csv"
if not FACE_CSV.is_file():
    raise FileNotFoundError(f"Missing face metadata CSV: {FACE_CSV}")
face_metadata = pd.read_csv(FACE_CSV)
print(face_metadata[["label", "category", "source", "fake_method"]].value_counts(dropna=False).head(20))
print("Face label mapping: REAL=0, FAKE=1; label_numeric is not used.")

# Face URLs are acquired only when this cell is run.
FACE_MANIFEST = build_face_manifest(
    FACE_CSV, FACE_ROOT / "sample_images", sample_per_class=SAMPLE_PER_CLASS, seed=SEED
)
AI_MANIFEST = build_ai_manifest(AI_EXTRACTED, sample_per_class=SAMPLE_PER_CLASS, seed=SEED)
MANIFESTS = {
    "face_dataset_external_test": FACE_MANIFEST,
    "ai_generated": AI_MANIFEST,
}
for task, manifest in MANIFESTS.items():
    print(task, len(manifest), manifest["label"].value_counts().sort_index().to_dict())
    manifest.to_csv(OUTPUT_ROOT / f"{task}_manifest.csv", index=False)

label  category      source       fake_method
FAKE   AI Generated  MultiSource  StyleGAN3      3767
REAL   Authentic     Unsplash     NaN            2790
Name: count, dtype: int64
Face label mapping: REAL=0, FAKE=1; label_numeric is not used.
face_dataset_external_test 200 {0: 100, 1: 100}
ai_generated 200 {0: 100, 1: 100}


## Preprocessing validation

In [14]:
sample_path = next(iter(FACE_MANIFEST["path"].tolist() + AI_MANIFEST["path"].tolist()), None)
if sample_path is None:
    raise RuntimeError("No valid evaluation image is available.")
with Image.open(sample_path) as source:
    sample_image = source.convert("RGB")
rgb = preprocess_rgb(sample_image)
fft = preprocess_fft(sample_image)
assert rgb.shape == (3, 224, 224) and rgb.dtype == np.float32
assert fft.shape == (3, 224, 224) and fft.dtype == np.float32
assert np.isfinite(rgb).all() and np.isfinite(fft).all()
print("RGB:", rgb.shape, rgb.dtype, "FFT:", fft.shape, fft.dtype)

RGB: (3, 224, 224) float32 FFT: (3, 224, 224) float32


## Checkpoint registry and compatibility

In [15]:
CHECKPOINTS = {
    "ai_generated": [
        ModelSpec("ai_generated", "rgb", PROJECT_ROOT / "experiments/rgb-baseline/cifake-image-v2/cifake-rgb-resnet18-best.pt", "cifake", "full_image_rgb"),
        ModelSpec("ai_generated", "fft", PROJECT_ROOT / "experiments/fft-baseline/cifake-image-v2/cifake-fft-resnet18-best.pt", "cifake", "full_image_fft"),
        ModelSpec("ai_generated", "fusion", PROJECT_ROOT / "experiments/fusion-baseline/cifake-image-v2/cifake-fusion-resnet18-best.pt", "cifake", "full_image_rgb_fft"),
    ],
    "face_dataset_external_test": [
        ModelSpec("face_dataset_external_test", "rgb", PROJECT_ROOT / "experiments/rgb-baseline/faceforensicspp-c23-r2-v1-limited-clean/faceforensicspp-rgb-resnet18-best.pt", "faceforensicspp", "full_image_rgb_exploratory"),
    ],
}
LOADED = {}
for task, specs in CHECKPOINTS.items():
    LOADED[task] = [load_evaluation_model(spec, DEVICE) for spec in specs]
    for item in LOADED[task]:
        print(task, item.spec.model_type, item.status, item.reason or "ready")

Skipping E:\Projects\Deepfake detector\experiments\rgb-baseline\cifake-image-v2\cifake-rgb-resnet18-best.pt: Checkpoint is not compatible with safe weights-only loading: E:\Projects\Deepfake detector\experiments\rgb-baseline\cifake-image-v2\cifake-rgb-resnet18-best.pt. Re-export it as a state-dict checkpoint offline.
Skipping E:\Projects\Deepfake detector\experiments\fft-baseline\cifake-image-v2\cifake-fft-resnet18-best.pt: Checkpoint is not compatible with safe weights-only loading: E:\Projects\Deepfake detector\experiments\fft-baseline\cifake-image-v2\cifake-fft-resnet18-best.pt. Re-export it as a state-dict checkpoint offline.
Skipping E:\Projects\Deepfake detector\experiments\fusion-baseline\cifake-image-v2\cifake-fusion-resnet18-best.pt: Checkpoint is not compatible with safe weights-only loading: E:\Projects\Deepfake detector\experiments\fusion-baseline\cifake-image-v2\cifake-fusion-resnet18-best.pt. Re-export it as a state-dict checkpoint offline.
Skipping E:\Projects\Deepfake d

ai_generated rgb skipped Checkpoint is not compatible with safe weights-only loading: E:\Projects\Deepfake detector\experiments\rgb-baseline\cifake-image-v2\cifake-rgb-resnet18-best.pt. Re-export it as a state-dict checkpoint offline.
ai_generated fft skipped Checkpoint is not compatible with safe weights-only loading: E:\Projects\Deepfake detector\experiments\fft-baseline\cifake-image-v2\cifake-fft-resnet18-best.pt. Re-export it as a state-dict checkpoint offline.
ai_generated fusion skipped Checkpoint is not compatible with safe weights-only loading: E:\Projects\Deepfake detector\experiments\fusion-baseline\cifake-image-v2\cifake-fusion-resnet18-best.pt. Re-export it as a state-dict checkpoint offline.
face_dataset_external_test rgb skipped Checkpoint is not compatible with safe weights-only loading: E:\Projects\Deepfake detector\experiments\rgb-baseline\faceforensicspp-c23-r2-v1-limited-clean\faceforensicspp-rgb-resnet18-best.pt. Re-export it as a state-dict checkpoint offline.


## Inference, metrics, reports, and exports

Scores are softmax fake-class scores, not calibrated probabilities. Metrics are never combined across tasks.

In [16]:
RESULTS = {}
METRICS = []
for task, manifest in MANIFESTS.items():
    RESULTS[task] = {}
    for loaded in LOADED[task]:
        if loaded.model is None:
            continue
        result = predict_model(loaded, manifest, DEVICE, batch_size=BATCH_SIZE)
        RESULTS[task][loaded.spec.model_type] = result
        metrics = calculate_model_metrics(result)
        METRICS.append({"task": task, "model": loaded.spec.model_type, **metrics})
        export_results(task, loaded.spec.model_type, result, OUTPUT_ROOT, seed=SEED)
        (OUTPUT_ROOT / task / f"{loaded.spec.model_type}_bootstrap.json").write_text(
            json.dumps(bootstrap_intervals(result, seed=SEED), indent=2), encoding="utf-8"
        )
        (OUTPUT_ROOT / task / f"{loaded.spec.model_type}_classification_report.json").write_text(
            json.dumps(classification_report_dict(result), indent=2), encoding="utf-8"
        )
    if RESULTS[task]:
        plot_curves(task, RESULTS[task], OUTPUT_ROOT)
metrics_df = pd.DataFrame(METRICS)
display(metrics_df)
metrics_df.to_csv(OUTPUT_ROOT / "external_evaluation_metrics.csv", index=False)
(OUTPUT_ROOT / "external_evaluation_summary.json").write_text(
    json.dumps(METRICS, indent=2, default=str), encoding="utf-8"
)

""


2

## Confusion matrices and error analysis

In [18]:
for task, model_results in RESULTS.items():
    for model_name, result in model_results.items():
        cm = np.asarray(calculate_model_metrics(result)["confusion_matrix"])
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])
        plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"{task} / {model_name}")
        plt.tight_layout(); plt.savefig(OUTPUT_ROOT / task / f"{model_name}_confusion_matrix.png", dpi=160); plt.show(); plt.close()
        errors = pd.DataFrame({"path": result["paths"], "actual": result["labels"], "predicted": result["predictions"], "fake_score": result["scores"]})
        errors["correct"] = errors["actual"] == errors["predicted"]
        errors.to_csv(OUTPUT_ROOT / task / f"{model_name}_error_analysis.csv", index=False)
        print(task, model_name, "false positives:", int(((errors.actual == 0) & (errors.predicted == 1)).sum()), "false negatives:", int(((errors.actual == 1) & (errors.predicted == 0)).sum()))

## Final validation checklist

In [19]:
CHECKLIST = {
    "project_paths_initialized": PROJECT_ROOT.is_dir() and OUTPUT_ROOT.is_dir(),
    "face_metadata_loaded": FACE_CSV.is_file() and not face_metadata.empty,
    "ai_manifest_built": not AI_MANIFEST.empty,
    "both_class_labels_verified": all(set(m["label"]) == {0, 1} for m in MANIFESTS.values()),
    "balanced_manifests": all(m["label"].value_counts().nunique() == 1 for m in MANIFESTS.values()),
    "preprocessing_validated": rgb.shape == (3, 224, 224) and fft.shape == (3, 224, 224),
    "inference_completed": bool(RESULTS),
    "metrics_exported": (OUTPUT_ROOT / "external_evaluation_metrics.csv").is_file(),
    "summary_exported": (OUTPUT_ROOT / "external_evaluation_summary.json").is_file(),
}
for name, passed in CHECKLIST.items():
    print(f"{'PASS' if passed else 'FAIL':<5} {name}")

PASS  project_paths_initialized
PASS  face_metadata_loaded
PASS  ai_manifest_built
PASS  both_class_labels_verified
PASS  balanced_manifests
PASS  preprocessing_validated
PASS  inference_completed
PASS  metrics_exported
PASS  summary_exported
